# Lab 11: Principal Component Regression — Explaining Solar and Wind Production from ERA5 Weather Maps

**Course:** 1454 - Computing in Civil Engineering
**Degree:** Master's Degree in Civil Engineering (M.Sc.) — Universidad de Cantabria

---

### Engineering Motivation
In Lab 10, a model with trend + harmonics explained most of the daily variability of electricity **demand**, but only $R^2 \approx 0.15$ of daily **wind** production: wind depends on the **weather** (storms, anticyclones), not on the calendar.

Weather is available as **maps**: the ERA5 reanalysis gives, every day, the wind speed at 100 m and the solar radiation at ~250 grid points over the Iberian Peninsula. Using all grid points as predictors is a bad idea: neighbouring points are almost identical (**multicollinearity**) and we would fit hundreds of coefficients.

**Principal Component Analysis (PCA)** compresses each daily map into a few uncorrelated numbers, the **principal components (PCs)**, which describe the dominant weather patterns (the same idea as the NAO and EA indices of P09, which are themselves principal components of pressure maps). We then use the PCs as predictors in a linear regression: **Principal Component Regression (PCR)**.

### Datasets
* `era5_wind100_daily_spain.csv`: daily mean wind speed at 100 m (m/s); one row per day, one column per grid point.
* `era5_ssrd_daily_spain.csv`: daily surface solar radiation (kWh/m²/day); same structure.
* `electricity_demand_spain.csv`: hourly `Load`, `Solar` and `Wind` production in Spain (MW), from Lab 10.

### Working Plan (≈ 1 h 30 min)
* **Part 1 (≈ 20 min):** PCA of the solar and wind maps.
* **Part 2 (≈ 30 min):** regression of daily **solar** production on the solar PCs.
* **Part 3 (≈ 30 min):** regression of daily **wind** production on the wind PCs, and choice of the number of components.

---

### Quick Reference: Key Functions

| Function / Method | Syntax Example | Brief Description |
| :--- | :--- | :--- |
| **PCA object** | `pca = PCA(n_components=0.95)` | Keeps as many PCs as needed to explain 95% of the variance (or give an integer). |
| **Fit + transform** | `pcs = pca.fit_transform(X_train)` | Learns the patterns (EOFs) from `X_train` and returns its PCs. |
| **Transform only** | `pcs_new = pca.transform(X_new)` | Projects new data onto the patterns learned before. |
| **Number of PCs** | `pca.n_components_` | Number of components kept. |
| **Explained variance** | `pca.explained_variance_ratio_` | Fraction of the total variance explained by each PC. |
| **DataFrame of PCs** | `pd.DataFrame(pcs, index=dates, columns=[f"PC{i+1}" for i in range(k)])` | Named columns `PC1, PC2, …` to use as predictors. |
| **Join by date** | `daily[["Solar"]].join(pcs_df)` | Aligns production and PCs on the same days. |
| **OLS with a table of predictors** | `sm.OLS(y, sm.add_constant(X)).fit()` | Linear regression without a formula; `add_constant` adds the intercept. |
| **All columns but one** | `df.drop(columns="Solar")` | Predictors = every column except the predictand. |
| **Prediction** | `model.predict(sm.add_constant(X_new))` | Predicted values for new data. |
| **Serial correlation** | `durbin_watson(model.resid)` | ≈ 2: independent residuals; < 2: positive autocorrelation. |

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from sklearn.decomposition import PCA
from statsmodels.stats.stattools import durbin_watson

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

In [ ]:
# Provided: load the data
data_dir = "."      # folder with the three CSV files

solar_era5 = pd.read_csv(os.path.join(data_dir, "era5_ssrd_daily_spain.csv"), index_col=0, parse_dates=True)
wind_era5  = pd.read_csv(os.path.join(data_dir, "era5_wind100_daily_spain.csv"), index_col=0, parse_dates=True)

electricity = pd.read_csv(os.path.join(data_dir, "electricity_demand_spain.csv"), index_col=0, parse_dates=True)
daily = electricity.resample("D").sum() / 1000          # daily energy in GWh/day

print(f"ERA5 solar maps: {solar_era5.shape[0]} days x {solar_era5.shape[1]} grid points")
print(f"ERA5 wind maps:  {wind_era5.shape[0]} days x {wind_era5.shape[1]} grid points")
print(f"Electricity:     {len(daily)} days ({daily.index[0]:%Y-%m-%d} -> {daily.index[-1]:%Y-%m-%d})")

In [ ]:
# Provided: plot observed vs. fitted values and the residuals of a model
def plot_fit(observed, model, label="Production (GWh/day)"):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
    ax1.plot(observed, color="0.6", lw=0.8, label="Observed")
    ax1.plot(model.fittedvalues, color="#eb6834", lw=1.2, label=f"Fit (R² = {model.rsquared:.2f})")
    ax1.set_ylabel(label)
    ax1.legend(frameon=False, loc="upper left")
    ax2.plot(model.resid, color="k", lw=0.6)
    ax2.axhline(0, color="r", lw=1)
    ax2.set_ylabel("Residual")
    plt.tight_layout()
    plt.show()

---
## Part 1 — PCA of the Weather Maps

Each ERA5 file is a **data matrix** $X$ of size $n_{\text{days}} \times n_{\text{points}}$. PCA finds the spatial patterns (EOFs) that explain most of the day-to-day variance of the maps, and summarises each day by a few numbers: its principal components.

To be able to test the models later on a year **not used** to build them, the PCA must learn the patterns from the training years only (2015–2019) and then **project** all days onto those patterns.

### Tasks:
1. Create a PCA object for the solar maps that keeps the components needed to explain **95%** of the variance. Fit it on **2015–2019** (`solar_era5.loc[:"2019"]`) and transform **all** days (`pca.transform(solar_era5)`).
2. Store the result in a DataFrame `solar_pcs` with the same dates as `solar_era5` and columns `PC1, PC2, …`.
3. Repeat for the wind maps → `wind_pcs`.
4. How many components are needed in each case? How much variance do the first 3 PCs explain?

In [ ]:
# 1.1 PCA of the solar and wind maps (fit on 2015-2019, transform all days)

### Questions to Answer (Part 1)
1. How many components are needed to explain 95% of the variance of the solar maps? And of the wind maps? Which field is "simpler" (needs fewer patterns), and why could that be?
2. Why don't we regress the production directly on the ~250 grid points? What would happen to the coefficients?
3. Why must the PCA be fitted on 2015–2019 only, and 2020 just **projected** with `transform`?

*Write your answers to Part 1 here:*

1. ...
2. ...
3. ...

---
## Part 2 — Solar Production

### Tasks:
1. Join the daily `Solar` production with `solar_pcs` (`daily[["Solar"]].join(solar_pcs)`) and drop the days with missing values.
2. Add a trend: `time_num` (decimal years since the first year) and `time_num_sqr` = `time_num`$^2$.
3. Fit `Solar ~ all PCs + time_num + time_num_sqr` on **2015–2020** with `sm.OLS` (predictors: all columns except `Solar`). Plot it with `plot_fit`.
4. Refit the same model on **2019–2020** only (`y.loc["2019":]`, `X.loc["2019":]`) and plot it.
5. Compute the Durbin-Watson statistic of the residuals of the 2019–2020 model.

In [ ]:
# 2.1 Join Solar + PCs, add the trend and fit the model on 2015-2020

In [ ]:
# 2.2 Refit on 2019-2020, plot and compute the Durbin-Watson statistic

### Questions to Answer (Part 2)
4. The model fitted on 2015–2020 reaches a high $R^2$. Is it a good model? Look at the residuals over time: what pattern do you see, and what physical cause explains it?
5. Why does the model predict **negative** solar production on some winter days?
6. What changes in $R^2$ and in the residuals when you fit 2019–2020 only?
7. In the 2019–2020 model, the residuals are larger in summer and in 2020 than in winter. Why? What does it say about the relation *production = installed capacity × radiation*?
8. Are consecutive residuals independent? What does the Durbin-Watson statistic imply for the $p$-values of the PCs?

*Write your answers to Part 2 here:*

4. ...
5. ...
6. ...
7. ...
8. ...

---
## Part 3 — Wind Production

### Tasks:
1. Join the daily `Wind` production with `wind_pcs` and drop the days with missing values.
2. Fit `Wind ~ all wind PCs` on 2015–2020 with `sm.OLS` and plot it with `plot_fit`.
3. **How many components?** For $k = 1, 3, 5, 10, 15$ and all PCs, fit `Wind ~ PC1 + … + PCk` on **2015–2019** and predict **2020**. Build a table with $R^2$ (training), AIC and the **RMSE in 2020**.

In [ ]:
# 3.1 Join Wind + PCs, fit with all PCs and plot

In [ ]:
# 3.2 Number of components: train 2015-2019, test 2020 -> table with R² (train), AIC and RMSE 2020
# Hint: df_wind.columns[1:k+1] are the first k PCs

### Questions to Answer (Part 3)
9. In Lab 10, harmonics explained $R^2 \approx 0.15$ of daily wind energy. How much do the wind PCs explain? Why such a difference?
10. Why does $R^2$ (training) always increase with $k$? Which number of components would you choose, and why?
11. The largest errors are big **negative** spikes on a few days (e.g. late December 2019). What happened on those days, and why does a linear model in wind speed fail precisely there? (Hint: look up the *power curve* of a wind turbine.)
12. **Engineering conclusion:** can these models forecast the solar and wind production of **next year**? What would you need? Which model would you use for long-term planning (Lab 10) and which one for day-ahead operation?

*Write your answers to Part 3 here:*

9. ...
10. ...
11. ...
12. ...